# The Arabic Tokenizer Tax — Live Demo

**A runnable companion to the article _The Tokenizer Tax: The Hidden Cost of Arabic AI Workloads_.**

This notebook measures how many tokens the same content produces in English vs Arabic across several frontier and open-weight tokenizers, then projects what that gap looks like as a monthly bill on a realistic enterprise workload.

**How to use it:**
1. Click **Runtime → Run all** in the menu above. The first run takes about 60–90 seconds to install dependencies.
2. Read the comparison table and the cost projection.


Feedback, corrections, and PRs welcome: [github.com/lakshmigpnthn/arabic-tokenizer-tax](https://github.com/)

## 1. Install dependencies

We use `tiktoken` for OpenAI tokenizers (exact) and `transformers` for open-weight tokenizers (exact). No API keys or Hugging Face logins required for the default set.

In [16]:
%%capture
!pip install -q tiktoken transformers sentencepiece pandas

In [17]:
from huggingface_hub import login
login()

## 2. Load the tokenizers

We load:
- **OpenAI `cl100k_base`** — used by GPT-4 and the GPT-4o family. Exact.
- **OpenAI `o200k_base`** — used by newer GPT-4o variants. Exact.
- **Qwen 2.5** — open-weight, multilingual, no gating. Exact.
- **Llama 3** — open-weight, English-dominant training. Exact.
- **Jais** — open-weight, Arabic-native models. Exact.


In [18]:
import tiktoken
from transformers import AutoTokenizer

# OpenAI tokenizers — instant, no download
enc_cl100k = tiktoken.get_encoding('cl100k_base')
enc_o200k = tiktoken.get_encoding('o200k_base')

# Open-weight tokenizers — download once, cached afterwards
print('Loading Qwen 2.5 tokenizer...')
tok_qwen = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-7B', trust_remote_code=True)

print('Loading Llama 3 tokenizer...')
tok_llama = AutoTokenizer.from_pretrained('unsloth/llama-3-8b', trust_remote_code=True)

print('Loading Jais tokenizer...')
tok_jais = AutoTokenizer.from_pretrained('inceptionai/jais-13b', trust_remote_code=True)

print('Done.')

def count_tokens(text):
    return {
        'cl100k_base (GPT-4 / GPT-4o)': len(enc_cl100k.encode(text)),
        'o200k_base (newer GPT-4o)':    len(enc_o200k.encode(text)),
        'Qwen 2.5':                     len(tok_qwen.encode(text, add_special_tokens=False)),
        'Llama 3':                      len(tok_llama.encode(text, add_special_tokens=False)),
        'Jais 13b':                     len(tok_jais.encode(text, add_special_tokens=False)),
    }

Loading Qwen 2.5 tokenizer...
Loading Llama 3 tokenizer...
Loading Jais tokenizer...
Done.


## 3. Sample content

The texts below are an English suspicious-transaction (AML) review summary and its Arabic equivalent. Same business content, same compliance purpose — just rendered in two languages. This is realistic of the kind of content a Gulf bank's AI workflow would handle.

In [19]:
EN_TEXT = '''SUSPICIOUS TRANSACTION REVIEW — CASE SUMMARY

The subject account was flagged on 12 May for a series of high-value transfers that fall outside the customer's established transaction profile. Over a seven-day period, the account received six inbound wire transfers from counterparties in three different jurisdictions, totalling approximately USD 180,000. Within 24 to 48 hours of each receipt, an equivalent or near-equivalent amount was transferred to a single beneficiary account in a fourth jurisdiction.

The customer is classified as a small-business client with a stated business purpose of consumer electronics import and resale. The historical transaction pattern over the preceding twelve months shows monthly inflows averaging USD 35,000, with corresponding outflows distributed across multiple suppliers.

The flagged activity is inconsistent with the stated business purpose and customer profile in three respects: the velocity of the inbound transfers, the concentration of outbound transfers to a single beneficiary, and the geographic spread of counterparties relative to the customer's declared trading partners.

The recommended action is to file a suspicious activity report, place a temporary hold on the beneficiary account, and request additional documentation from the customer regarding the underlying commercial transactions. Compliance team review is required before any further account activity is processed.'''

AR_TEXT = '''\u0645\u0631\u0627\u062c\u0639\u0629 \u0645\u0639\u0627\u0645\u0644\u0629 \u0645\u0634\u0628\u0648\u0647\u0629 \u2014 \u0645\u0644\u062e\u0635 \u0627\u0644\u062d\u0627\u0644\u0629

\u062a\u0645 \u0648\u0636\u0639 \u0639\u0644\u0627\u0645\u0629 \u0639\u0644\u0649 \u0627\u0644\u062d\u0633\u0627\u0628 \u0627\u0644\u0645\u0639\u0646\u064a \u0641\u064a \u0627\u0644\u062b\u0627\u0646\u064a \u0639\u0634\u0631 \u0645\u0646 \u0645\u0627\u064a\u0648 \u0628\u0633\u0628\u0628 \u0633\u0644\u0633\u0644\u0629 \u0645\u0646 \u0627\u0644\u062a\u062d\u0648\u064a\u0644\u0627\u062a \u0639\u0627\u0644\u064a\u0629 \u0627\u0644\u0642\u064a\u0645\u0629 \u0627\u0644\u062a\u064a \u062a\u0642\u0639 \u062e\u0627\u0631\u062c \u0646\u0637\u0627\u0642 \u0645\u0644\u0641 \u0627\u0644\u0645\u0639\u0627\u0645\u0644\u0627\u062a \u0627\u0644\u0645\u0639\u062a\u0627\u062f \u0644\u0644\u0639\u0645\u064a\u0644. \u0639\u0644\u0649 \u0645\u062f\u0649 \u0641\u062a\u0631\u0629 \u0633\u0628\u0639\u0629 \u0623\u064a\u0627\u0645\u060c \u062a\u0644\u0642\u0649 \u0627\u0644\u062d\u0633\u0627\u0628 \u0633\u062a\u0629 \u062a\u062d\u0648\u064a\u0644\u0627\u062a \u0628\u0631\u0642\u064a\u0629 \u0648\u0627\u0631\u062f\u0629 \u0645\u0646 \u0623\u0637\u0631\u0627\u0641 \u0645\u0642\u0627\u0628\u0644\u0629 \u0641\u064a \u062b\u0644\u0627\u062b \u0648\u0644\u0627\u064a\u0627\u062a \u0642\u0636\u0627\u0626\u064a\u0629 \u0645\u062e\u062a\u0644\u0641\u0629\u060c \u0628\u0625\u062c\u0645\u0627\u0644\u064a \u064a\u0628\u0644\u063a \u062d\u0648\u0627\u0644\u064a 180\u060c000 \u062f\u0648\u0644\u0627\u0631 \u0623\u0645\u0631\u064a\u0643\u064a. \u062e\u0644\u0627\u0644 24 \u0625\u0644\u0649 48 \u0633\u0627\u0639\u0629 \u0645\u0646 \u0643\u0644 \u0625\u064a\u062f\u0627\u0639\u060c \u062a\u0645 \u062a\u062d\u0648\u064a\u0644 \u0645\u0628\u0644\u063a \u0645\u0633\u0627\u0648\u064d \u0623\u0648 \u0642\u0631\u064a\u0628 \u0645\u0646\u0647 \u0625\u0644\u0649 \u062d\u0633\u0627\u0628 \u0645\u0633\u062a\u0641\u064a\u062f \u0648\u0627\u062d\u062f \u0641\u064a \u0648\u0644\u0627\u064a\u0629 \u0642\u0636\u0627\u0626\u064a\u0629 \u0631\u0627\u0628\u0639\u0629.

\u064a\u062a\u0645 \u062a\u0635\u0646\u064a\u0641 \u0627\u0644\u0639\u0645\u064a\u0644 \u0643\u0639\u0645\u064a\u0644 \u0623\u0639\u0645\u0627\u0644 \u0635\u063a\u064a\u0631\u0629 \u0645\u0639 \u063a\u0631\u0636 \u062a\u062c\u0627\u0631\u064a \u0645\u0639\u0644\u0646 \u0644\u0627\u0633\u062a\u064a\u0631\u0627\u062f \u0648\u0625\u0639\u0627\u062f\u0629 \u0628\u064a\u0639 \u0627\u0644\u0625\u0644\u0643\u062a\u0631\u0648\u0646\u064a\u0627\u062a \u0627\u0644\u0627\u0633\u062a\u0647\u0644\u0627\u0643\u064a\u0629. \u064a\u0638\u0647\u0631 \u0646\u0645\u0637 \u0627\u0644\u0645\u0639\u0627\u0645\u0644\u0627\u062a \u0627\u0644\u062a\u0627\u0631\u064a\u062e\u064a \u062e\u0644\u0627\u0644 \u0627\u0644\u0627\u062b\u0646\u064a \u0639\u0634\u0631 \u0634\u0647\u0631\u064b\u0627 \u0627\u0644\u0633\u0627\u0628\u0642\u0629 \u062a\u062f\u0641\u0642\u0627\u062a \u0634\u0647\u0631\u064a\u0629 \u062f\u0627\u062e\u0644\u064a\u0629 \u062a\u0628\u0644\u063a \u0641\u064a \u0627\u0644\u0645\u062a\u0648\u0633\u0637 35\u060c000 \u062f\u0648\u0644\u0627\u0631 \u0623\u0645\u0631\u064a\u0643\u064a\u060c \u0645\u0639 \u062a\u062f\u0641\u0642\u0627\u062a \u062e\u0627\u0631\u062c\u064a\u0629 \u0645\u0642\u0627\u0628\u0644\u0629 \u0645\u0648\u0632\u0639\u0629 \u0639\u0644\u0649 \u0639\u062f\u0629 \u0645\u0648\u0631\u062f\u064a\u0646.

\u0627\u0644\u0646\u0634\u0627\u0637 \u0627\u0644\u0645\u0648\u0633\u0648\u0645 \u0644\u0627 \u064a\u062a\u0633\u0642 \u0645\u0639 \u0627\u0644\u063a\u0631\u0636 \u0627\u0644\u062a\u062c\u0627\u0631\u064a \u0627\u0644\u0645\u0639\u0644\u0646 \u0648\u0645\u0644\u0641 \u0627\u0644\u0639\u0645\u064a\u0644 \u0645\u0646 \u062b\u0644\u0627\u062b\u0629 \u062c\u0648\u0627\u0646\u0628: \u0633\u0631\u0639\u0629 \u0627\u0644\u062a\u062d\u0648\u064a\u0644\u0627\u062a \u0627\u0644\u0648\u0627\u0631\u062f\u0629\u060c \u0648\u062a\u0631\u0643\u064a\u0632 \u0627\u0644\u062a\u062d\u0648\u064a\u0644\u0627\u062a \u0627\u0644\u0635\u0627\u062f\u0631\u0629 \u0625\u0644\u0649 \u0645\u0633\u062a\u0641\u064a\u062f \u0648\u0627\u062d\u062f\u060c \u0648\u0627\u0644\u0627\u0646\u062a\u0634\u0627\u0631 \u0627\u0644\u062c\u063a\u0631\u0627\u0641\u064a \u0644\u0644\u0623\u0637\u0631\u0627\u0641 \u0627\u0644\u0645\u0642\u0627\u0628\u0644\u0629 \u0645\u0642\u0627\u0631\u0646\u0629 \u0628\u0627\u0644\u0634\u0631\u0643\u0627\u0621 \u0627\u0644\u062a\u062c\u0627\u0631\u064a\u064a\u0646 \u0627\u0644\u0645\u0639\u0644\u0646\u064a\u0646 \u0644\u0644\u0639\u0645\u064a\u0644.

\u0627\u0644\u0625\u062c\u0631\u0627\u0621 \u0627\u0644\u0645\u0648\u0635\u0649 \u0628\u0647 \u0647\u0648 \u062a\u0642\u062f\u064a\u0645 \u062a\u0642\u0631\u064a\u0631 \u0646\u0634\u0627\u0637 \u0645\u0634\u0628\u0648\u0647\u060c \u0648\u0648\u0636\u0639 \u062a\u062c\u0645\u064a\u062f \u0645\u0624\u0642\u062a \u0639\u0644\u0649 \u062d\u0633\u0627\u0628 \u0627\u0644\u0645\u0633\u062a\u0641\u064a\u062f\u060c \u0648\u0637\u0644\u0628 \u0648\u062b\u0627\u0626\u0642 \u0625\u0636\u0627\u0641\u064a\u0629 \u0645\u0646 \u0627\u0644\u0639\u0645\u064a\u0644 \u062d\u0648\u0644 \u0627\u0644\u0645\u0639\u0627\u0645\u0644\u0627\u062a \u0627\u0644\u062a\u062c\u0627\u0631\u064a\u0629 \u0627\u0644\u0623\u0633\u0627\u0633\u064a\u0629. \u0645\u0631\u0627\u062c\u0639\u0629 \u0641\u0631\u064a\u0642 \u0627\u0644\u0627\u0645\u062a\u062b\u0627\u0644 \u0645\u0637\u0644\u0648\u0628\u0629 \u0642\u0628\u0644 \u0645\u0639\u0627\u0644\u062c\u0629 \u0623\u064a \u0646\u0634\u0627\u0637 \u0625\u0636\u0627\u0641\u064a \u0644\u0644\u062d\u0633\u0627\u0628.'''

print(f'English: {len(EN_TEXT):,} characters')
print(f'Arabic:  {len(AR_TEXT):,} characters')

English: 1,434 characters
Arabic:  1,115 characters


## 4. Run the comparison

Note: **character counts are similar between English and Arabic in this sample**. The interesting question is what happens at the token level, which is what you actually pay for.

In [20]:
import pandas as pd

en_counts = count_tokens(EN_TEXT)
ar_counts = count_tokens(AR_TEXT)

rows = []
for name in en_counts:
    en = en_counts[name]
    ar = ar_counts[name]
    rows.append({
        'Tokenizer': name,
        'EN tokens': en,
        'AR tokens': ar,
        'AR/EN ratio': round(ar / en, 2),
    })

df = pd.DataFrame(rows)
df

,Tokenizer,EN tokens,AR tokens,AR/EN ratio
0,cl100k_base (GPT-4 / GPT-4o),242,743,3.07
1,o200k_base (newer GPT-4o),241,330,1.37
2,Qwen 2.5,252,405,1.61
3,Llama 3,242,403,1.67
4,Jais 13b,264,234,0.89


## 5. Project the monthly cost

Take the measured ratio and apply it to a realistic enterprise workload — a UAE bank running 500,000 AML alerts a month, each producing roughly 2,000 input tokens of context and 500 output tokens of response in the English baseline.

**Pricing here is illustrative.** Update with your contracted rates or current published rates before quoting any specific figure to a CFO.

In [21]:
# --- Workload (edit these to match your case) ---
VOLUME_PER_MONTH = 500_000
EN_INPUT_TOKENS_PER_CALL = 2_000
EN_OUTPUT_TOKENS_PER_CALL = 500

# --- Pricing in USD per 1M tokens (illustrative) ---
PRICING = {
    'GPT-4o':           {'input': 2.50, 'output': 10.00},
    'Claude Sonnet 4':  {'input': 3.00, 'output': 15.00},
    'Gemini 1.5 Pro':   {'input': 1.25, 'output':  5.00},
}

# Use cl100k_base ratio as the reference (you can switch this to any tokenizer above)
RATIO = ar_counts['cl100k_base (GPT-4 / GPT-4o)'] / en_counts['cl100k_base (GPT-4 / GPT-4o)']
print(f'AR/EN ratio (cl100k_base): {RATIO:.2f}x\n')

ar_input = EN_INPUT_TOKENS_PER_CALL * RATIO
ar_output = EN_OUTPUT_TOKENS_PER_CALL * RATIO

cost_rows = []
for model, p in PRICING.items():
    en_cost = (
        (EN_INPUT_TOKENS_PER_CALL * VOLUME_PER_MONTH / 1_000_000) * p['input']
        + (EN_OUTPUT_TOKENS_PER_CALL * VOLUME_PER_MONTH / 1_000_000) * p['output']
    )
    ar_cost = (
        (ar_input * VOLUME_PER_MONTH / 1_000_000) * p['input']
        + (ar_output * VOLUME_PER_MONTH / 1_000_000) * p['output']
    )
    cost_rows.append({
        'Model': model,
        'EN monthly ($)': round(en_cost, 0),
        'AR monthly ($)': round(ar_cost, 0),
        'Delta ($)':      round(ar_cost - en_cost, 0),
        'Delta (%)':      round((ar_cost - en_cost) / en_cost * 100, 0),
    })

pd.DataFrame(cost_rows)

AR/EN ratio (cl100k_base): 3.07x



,Model,EN monthly ($),AR monthly ($),Delta ($),Delta (%)
0,GPT-4o,5000.0,15351.0,10351.0,207.0
1,Claude Sonnet 4,6750.0,20724.0,13974.0,207.0
2,Gemini 1.5 Pro,2500.0,7676.0,5176.0,207.0


## What this notebook does and does not prove

**It does show:**
- That equivalent content tokenizes very differently across English and Arabic on frontier tokenizers.
- That the cost gap scales linearly with workload volume.
- That you can measure this on your own corpus in under a minute.

**It does not show:**
- The quality of the model's Arabic output — that's a separate evaluation.
- The latency or throughput implications — also separate.
- A vendor-by-vendor recommendation — pick what fits your accuracy, sovereignty, and cost constraints.

Use the numbers from this notebook to start a conversation with your vendor or your CFO, not to end one.

---

**Feedback wanted.** If you ran this on your own data and got surprising results, open an issue on the repo. If the methodology has a flaw, even better — tell me about it.